**Municipal Assistant with RAG**

Project-Specific Information
Project Description

Problem: Finding out which permit is required, which documents must be submitted, or how much a municipal service costs often requires visiting the municipal office in person. The information exists, but it is scattered across different sources and written in complex administrative language.

Solution: A RAG-based assistant that answers real questions about municipal services in plain language and with visible source references. If the information does not exist in the available data, the assistant explicitly states that it does not know and provides the Municipality's contact information instead of generating an unsupported answer.

Source Data (public): Municipality portal – https://www.cm-guimaraes.pt/servicos-online/formularios




**0. Installation**

In [1]:
%pip install "langchain==0.3.27" "langchain-community==0.3.31" "langchain-chroma==0.2.6" "langchain-core>=0.3.0,<0.4.0" "langchain-google-genai>=2.1,<3" "langchain-openai" sentence-transformers pypdf beautifulsoup4 requests gradio reportlab -qqq

# Forçar atualização destas dependências para evitar conflitos de importação no Python 3.13
%pip install --upgrade huggingface_hub transformers sentence-transformers -qqq

%pip install gTTS faster-whisper -qqq
%pip install sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.20.2 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 1.16.1 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 1.16.1 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
Using cached huggingface_hub-2.2.0-py3-non

**1. Setup**

In [2]:
import os, re, json, time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import gradio as gr
from google.colab import userdata

from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
# ==================================================
# API keys — M.E.L. (DeepSeek)
# ==================================================
# No Google Colab, adiciona DEEPSEEK_API_KEY em Secrets (ícone da chave).
# Não escrevas nem publiques a chave diretamente no notebook.
from google.colab import userdata

DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError(
        "Falta configurar DEEPSEEK_API_KEY em Colab > Secrets. "
        "Ativa o acesso ao notebook e volta a executar esta célula."
    )

os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY.strip()
print("Chave do M.E.L. carregada com segurança.")


Chave do M.E.L. carregada com segurança.


In [4]:
BASE_URL = "https://www.cm-guimaraes.pt"
LISTING_URL = f"{BASE_URL}/servicos-online/formularios"
PROC_MARKER = "detalhe-de-procedimento"   # só os links com isto são procedimentos reais

HEADERS = {"User-Agent": "Mozilla/5.0 (projeto Hack4Guimaraes 2026 - assistente RAG)"}

MAX_PROCEDURES = 60        # nº de procedimentos a ingerir (None = todos)
REQUEST_DELAY = 0.5
CACHE_FILE = "procedimentos_guimaraes.json"
CHROMA_DIR = "./chroma_guimaraes"
COLLECTION = "m.e.l_rag_db"

# Score mínimo de relevância (0 a 1, distância cosseno). Se responder "não sei" demasiado, baixa; se responder fora do tema, sobe.
MIN_SCORE = 0.20

# O modelo local (SmolLM2) ficou no Anexo, desligado por defeito. Põe True se quiseres carregá-lo (precisa de GPU).
USAR_MODELO_LOCAL = False


INFO_GERAL = """Informação geral sobre os formulários da Câmara Municipal de Guimarães.
Nos Serviços Online é possível preencher online os formulários, divididos por áreas.
O preenchimento online só é possível se o munícipe estiver registado e autenticado.
Dúvidas no preenchimento dos formulários: contactar o serviço responsável.
Telefone: 253 421 200 (chamada para a rede fixa nacional). E-mail: geral@cm-guimaraes.pt.
Morada: Largo Cónego José Maria Gomes, 4804-534 Guimarães."""

# Linhas de menu/cookies que aparecem em todas as páginas e só poluem a pesquisa
BOILERPLATE = [
    "Passar para o Conteúdo Principal", "Utilizamos cookies", "Personalizar cookies",
    "Aceitar todos os Cookies", "Voltar ao topo da página",
]

# Layer 1 - Ingestion
(syntax reviewed by an AI agent, created using Copilot Agent Builder)

### Fase 1: Ingestão de Dados e Construção do Catálogo

Nesta fase inicial, o assistente acede ao portal de formulários da Câmara Municipal de Guimarães para mapear os procedimentos e serviços disponíveis. Para evitar sobrecarregar o servidor municipal, implementamos limites e pequenos atrasos entre cada pedido (`REQUEST_DELAY`).

In [5]:
# ==================================================
# Folder Structure
# ==================================================

DATA_DIR = "data"

RAW_HTML_DIR = f"{DATA_DIR}/raw/html"
RAW_PDF_DIR = f"{DATA_DIR}/raw/pdfs"

PROCESSED_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_HTML_DIR, exist_ok=True)
os.makedirs(RAW_PDF_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Structure created.")

Structure created.


In [6]:
# ==================================================
# Utilities
# ==================================================

def clean_text(text):
    if not text:
        return ""

    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    linhas = [l.strip() for l in text.splitlines()]
    linhas = [l for l in linhas if l and not any(b in l for b in BOILERPLATE)]

    return "\n".join(linhas).strip()


def absolute_url(link):
    return urljoin(BASE_URL, link)

In [7]:
_HTML_CACHE = {}   # evita pedir a mesma página duas vezes (texto + PDFs)

def fetch_html(url):
    """Descarrega o conteúdo HTML garantindo a codificação correta dos caracteres."""
    if url in _HTML_CACHE:
        return _HTML_CACHE[url]
    try:
        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )
        r.raise_for_status()
        # Forçar UTF-8 para evitar problemas de caracteres portugueses desconfigurados
        if r.encoding == 'ISO-8859-1':
            r.encoding = 'utf-8'
        _HTML_CACHE[url] = r.text
        return r.text
    except Exception as e:
        print(f"ERRO ao aceder a {url}: {e}")
        return None

In [8]:
def discover_services():
    """Lê a listagem e devolve os procedimentos (com a área a que pertencem).
    1.º tenta só os links de procedimentos; se não houver, usa o filtro de links da versão anterior."""
    html = fetch_html(LISTING_URL)
    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")
    services = {}

    # --- Caminho principal: só links de procedimentos
    for a in soup.find_all("a", href=True):
        if PROC_MARKER not in a["href"]:
            continue
        title = a.get_text(" ", strip=True)
        if len(title) < 4:
            continue
        if title.isupper():
            title = title.capitalize()

        h = a.find_previous(["h2", "h3", "h4", "h5"])
        area = h.get_text(" ", strip=True) if h else ""
        url = absolute_url(a["href"])
        services[url] = {"title": title, "url": url, "area": area}

    if services:
        services = list(services.values())
        print(f"{len(services)} procedimentos encontrados")
        return services

    # --- Plano B: filtro de links institucionais (da versão anterior do notebook)
    print("AVISO: não encontrei links de procedimentos. A usar o filtro de links institucionais.")
    ignorar_padroes = [
        "accessibility", "cookie", "topo-da-pagina", "sticky", "heraldica",
        "camara-municipal", "assembleia-municipal", "freguesias", "eleicoes",
        "facebook", "twitter", "instagram", "youtube", "linkedin"
    ]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(" ", strip=True)
        if len(title) < 4:
            continue
        if href.startswith("#") or any(p in href.lower() or p in title.lower() for p in ignorar_padroes):
            continue
        url = absolute_url(href)
        services[url] = {"title": title, "url": url, "area": ""}

    services = list(services.values())
    print(f"{len(services)} serviços de formulários válidos encontrados após filtragem.")
    return services

In [9]:
services = discover_services()

services[:10]

54 procedimentos encontrados


[{'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0FTQ0VOU09SI0FTQ0VO',
  'area': 'ascensores'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0FTQ0VOU09SI0FTQ0VOU09S',
  'area': 'ATENDIMENTO'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLURFRjEjQ0EtREVGMQ==',
  'area': 'candidatura a apoio à pessoa com deficiência'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLUlETyNDQS1JRE8=',
  'area': 'cartão municipal sénior'},
 {'title': 'Atendimento',
  'url': 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NFTSNDRU0wMQ==',
  'area': 'cemitérios municipais'},
 {'title': 'Atendimento',
  'url

In [10]:
# ==================================================
# Download HTML
# ==================================================

def extract_page_content(url):
    html = fetch_html(url)
    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")

    # Remover elementos sem informação útil
    for tag in soup([
        "script", "style", "noscript",
        "header", "footer", "nav", "aside"
    ]):
        tag.decompose()

    # Encontrar o conteúdo principal
    scope = (
        soup.find("main")
        or soup.find("article")
        or soup.find("body")
        or soup
    )

    # Extrair texto em linhas separadas
    text = scope.get_text("\n", strip=True)
    text = clean_text(text)

    # Guardar o título real do procedimento no conteúdo
    h1 = soup.find("h1")
    title = (
        h1.get_text(" ", strip=True)
        if h1 else ""
    )

    if title and not text.startswith(title):
        text = f"Título: {title}\n{text}"

    return text


In [11]:
# ==================================================
# Detect PDFs
# ==================================================

def detect_pdfs(url):

    html = fetch_html(url)

    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")

    pdfs = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if ".pdf" in href.lower():

            pdfs.append({
                "title": a.get_text(" ", strip=True),
                "url": absolute_url(href)
            })

    return pdfs

In [12]:
catalog = []
services = discover_services()

alvo = services if MAX_PROCEDURES is None else services[:MAX_PROCEDURES]

for i, service in enumerate(alvo):
    print(f"[{i+1}/{len(alvo)}] A recolher: {service['title']}")
    text = extract_page_content(service["url"])
    pdfs = detect_pdfs(service["url"])

    catalog.append({
        "id": f"SERV_{i+1:03}",
        "title": service["title"],
        "area": service.get("area", ""),
        "url": service["url"],
        "text": text,
        "pdfs": pdfs,
        "retrieved": time.strftime("%Y-%m-%d")
    })
    time.sleep(REQUEST_DELAY)

print(f"Catálogo atualizado com {len(catalog)} serviços enriquecidos.")

54 procedimentos encontrados
[1/54] A recolher: Atendimento
[2/54] A recolher: Atendimento
[3/54] A recolher: Atendimento
[4/54] A recolher: Atendimento
[5/54] A recolher: Atendimento
[6/54] A recolher: Atendimento
[7/54] A recolher: Atendimento
[8/54] A recolher: Atendimento
[9/54] A recolher: Atendimento
[10/54] A recolher: Atendimento
[11/54] A recolher: Atendimento
[12/54] A recolher: Atendimento
[13/54] A recolher: Atendimento
[14/54] A recolher: Atendimento
[15/54] A recolher: Atendimento
[16/54] A recolher: Atendimento
[17/54] A recolher: Atendimento
[18/54] A recolher: Atendimento
[19/54] A recolher: Atendimento
[20/54] A recolher: Atendimento
[21/54] A recolher: Atendimento
[22/54] A recolher: Atendimento
[23/54] A recolher: Atendimento
[24/54] A recolher: Atendimento
[25/54] A recolher: Atendimento
[26/54] A recolher: Atendimento
[27/54] A recolher: Atendimento
[28/54] A recolher: Atendimento
[29/54] A recolher: Atendimento
[30/54] A recolher: Atendimento
[31/54] A recolher: 

In [13]:
# ==================================================
# Save Catalog
# ==================================================

CATALOG_FILE = f"{PROCESSED_DIR}/services_catalog.json"

with open(CATALOG_FILE, "w", encoding="utf-8") as f:

    json.dump(
        catalog,
        f,
        ensure_ascii=False,
        indent=2
    )

print(CATALOG_FILE)

data/processed/services_catalog.json


In [14]:
# ==================================================
# Ingestion Statistics
# ==================================================

total_chars = sum(
len(item["text"] or "")
for item in catalog
)

total_pdfs = sum(
len(item["pdfs"])
for item in catalog
)

print("Serviços:", len(catalog))
print("PDFs:", total_pdfs)
print("Caracteres:", total_chars)

Serviços: 54
PDFs: 22
Caracteres: 57226


**Data Ingestion**

In [15]:
import pandas as pd

def validar_ingestao(catalogo):
    obrigatorios = {"title", "url", "text", "pdfs"}
    erros = []
    for i, item in enumerate(catalogo):
        faltam = obrigatorios - set(item.keys())
        if faltam:
            erros.append({"registo": i, "erro": f"Campos em falta: {', '.join(sorted(faltam))}"})
        if not str(item.get("url", "")).startswith("http"):
            erros.append({"registo": i, "erro": "URL em falta ou inválido"})
        if not str(item.get("title", "")).strip():
            erros.append({"registo": i, "erro": "Título vazio"})
        if not str(item.get("text", "")).strip() and not item.get("pdfs"):
            erros.append({"registo": i, "erro": "Sem texto extraído e sem PDFs detetados"})
    urls = [str(x.get("url", "")).strip() for x in catalogo if x.get("url")]
    duplicados = sorted({u for u in urls if urls.count(u) > 1})
    resumo = {"total_registos": len(catalogo), "URLs_duplicados": len(duplicados),
              "registos_com_erro": len({e['registo'] for e in erros}),
              "total_erros": len(erros)}
    return resumo, pd.DataFrame(erros), duplicados

resumo_ingestao, erros_ingestao, urls_duplicados = validar_ingestao(catalog)
print("Resumo da ingestão:", resumo_ingestao)
display(erros_ingestao.head(30))
if urls_duplicados:
    print("URLs duplicados (primeiros 20):", urls_duplicados[:20])

Resumo da ingestão: {'total_registos': 54, 'URLs_duplicados': 0, 'registos_com_erro': 0, 'total_erros': 0}


""


In [16]:
def validar_tabela(df, nome, colunas_obrigatorias=None, chave=None):
    colunas_obrigatorias = colunas_obrigatorias or []
    problemas = []
    faltam = sorted(set(colunas_obrigatorias) - set(df.columns))
    if faltam:
        problemas.append(f"{nome}: faltam colunas {faltam}")
    for coluna in colunas_obrigatorias:
        if coluna in df.columns:
            nulos = int(df[coluna].isna().sum())
            if nulos:
                problemas.append(f"{nome}: {nulos} valores nulos em '{coluna}'")
    if chave and chave in df.columns:
        duplicados = int(df[chave].duplicated().sum())
        if duplicados:
            problemas.append(f"{nome}: {duplicados} valores duplicados na chave '{chave}'")
    return {"tabela": nome, "linhas": len(df), "colunas": len(df.columns),
            "estado": "OK" if not problemas else "VERIFICAR", "problemas": problemas}

# Exemplo de utilização (descomenta e adapta quando as tabelas estiverem disponíveis):
 #resultado_1 = validar_tabela(tabela_1, "Tabela 1", colunas_obrigatorias=["id", "title"], chave="id")
 #resultado_2 = validar_tabela(tabela_2, "Tabela 2", colunas_obrigatorias=["id", "url"], chave="id")
 #print(f"{resultado_1}\n# {resultado_2}")


In [17]:
documents = [
    Document(page_content=item["text"],
             metadata={"source": item["url"], "title": item["title"], "area": item.get("area", "")})
    for item in catalog if item["text"]
]

# informação geral e contactos da Câmara também fazem parte da base de conhecimento
documents.append(Document(
    page_content=INFO_GERAL,
    metadata={"source": LISTING_URL, "title": "Informação geral e contactos", "area": ""},
))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Cada chunk começa com o nome do serviço, para a pesquisa e o LLM saberem de que procedimento se trata
for c in chunks:
    cab = f"Serviço: {c.metadata['title']}"
    if c.metadata.get("area"):
        cab += f" (área: {c.metadata['area']})"
    c.page_content = cab + "\n" + c.page_content

print(f"{len(chunks)} chunks gerados a partir de {len(documents)} documentos")

96 chunks gerados a partir de 55 documentos


In [18]:
from langchain_chroma import Chroma
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Modelo de embeddings multilingue (percebe bem português; o all-MiniLM-L6-v2 foi pensado para inglês)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Se voltares a correr esta célula, apaga a coleção anterior para não duplicar chunks
try:
    Chroma(collection_name=COLLECTION, embedding_function=embeddings).delete_collection()
except Exception:
    pass

# 3. Recriar a base vetorial Chroma com os chunks gerados anteriormente
#    (distância cosseno: faz com que os scores fiquem entre 0 e 1 e o MIN_SCORE faça sentido)
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
)

# 4. Configurar o retriever para ir buscar os 3 chunks mais relevantes
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Base vetorial e retriever criados localmente com sucesso e a custo zero!")

/tmp/ipykernel_4945/638167629.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base vetorial e retriever criados localmente com sucesso e a custo zero!


Inference Phase


### Fase 2: Configuração do Modelo de Linguagem Local (LLM)

Devido a limitações comuns de quota e chaves em APIs comerciais (como OpenAI ou Gemini), optámos por carregar o modelo de linguagem **SmolLM2-1.7B-Instruct** diretamente no ambiente do Colab. Este modelo corre de forma 100% local, privada e gratuita, utilizando aceleração por GPU se disponível.

In [19]:
NAO_SEI = ("Não encontrei essa informação nos serviços e formulários do Município que tenho disponíveis. "
           "Para não lhe dar uma resposta errada, sugiro que contacte a Câmara Municipal de Guimarães: "
           "telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.")

print("Mensagem padrão definida.")

Mensagem padrão definida.


In [20]:
# ==================================================
# M.E.L. — LLM + PROMPT + CHAIN (DeepSeek)
# ==================================================

import os, time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError("Configura DEEPSEEK_API_KEY em Colab > Secrets e executa a célula de setup.")

# A API da DeepSeek é compatível com o formato OpenAI.
# "deepseek-flash" é o identificador atual; "deepseek-chat" fica como alternativa.
# O modo "thinking" vem ligado por defeito: desligamo-lo para as respostas serem mais rápidas e baratas.
MODELOS_DEEPSEEK = ["deepseek-flash", "deepseek-chat"]

llm = None
for MODELO in MODELOS_DEEPSEEK:
    try:
        _candidato = ChatOpenAI(
            model=MODELO,
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com",
            temperature=0.2,
            max_retries=2,
            timeout=60,
            extra_body={"thinking": {"type": "disabled"}},
        )
        _candidato.invoke("Diz apenas: OK")
        llm = _candidato
        break
    except Exception as e:
        print(f"  {MODELO}: não funcionou ({type(e).__name__}: {str(e)[:150]})")

if llm is None:
    raise RuntimeError("Nenhum modelo da DeepSeek respondeu. Verifica a chave e o saldo da conta.")

prompt = ChatPromptTemplate.from_template("""
És o M.E.L., o assistente do Balcão Aberto ("Falar com a Câmara sem burocracia"),
para os serviços municipais de Guimarães.

Regras:
1. Responde APENAS com base no contexto fornecido. Não inventes valores, prazos, documentos nem leis.
2. Usa linguagem simples, frases curtas e português de Portugal.
3. Quando a pergunta for "como faço...", responde em passos numerados.
4. Se o contexto não contiver a resposta, diz claramente que não sabes e direciona para o contacto geral:
   telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.

Contexto:
{context}

Pergunta:
{question}

Resposta:
""")

chain = prompt | llm | StrOutputParser()
print(f"✅ M.E.L. configurado com o modelo {MODELO}.")

✅ M.E.L. configurado com o modelo deepseek-flash.


In [21]:
# Teste rápido do M.E.L.: se isto falhar, o erro aparece aqui em vez de ficar escondido mais à frente
try:
    response = llm.invoke("Diz apenas: Teste")
    print("M.E.L. OK:", response.content)
except Exception as e:
    print("ERRO no M.E.L.:", type(e).__name__)
    print(str(e)[:500])

M.E.L. OK: Teste


 **Pipeline RAG**

### Fase 3: Pipeline RAG (Retrieval-Augmented Generation)

Aqui unimos a base de dados vetorial ao nosso modelo local. Quando um munícipe faz uma pergunta, o ChromaDB pesquisa as fontes mais parecidas. Se o nível de semelhança for baixo (abaixo de `MIN_SCORE`), o assistente devolve uma mensagem padrão amigável sem inventar respostas.

In [22]:
def formatar_resposta(resultado):
    """Formata o dicionário retornado pelo RAG numa string elegante para o Gradio."""
    texto = resultado["answer"]
    if resultado["sources"]:
        texto += "\n\n**Fontes:**\n" + "\n".join(f"- [{t}]({u})" for t, u in resultado["sources"])
    return texto

def rag_answer(question):
    """Devolve {"answer": texto, "sources": [(título, url), ...]}.
    Consome o M.E.L. (DeepSeek) de forma otimizada para respostas rápidas e econômicas.
    """
    # k=3 para diminuir o tamanho do contexto e poupar tokens
    resultados = vectordb.similarity_search_with_relevance_scores(question, k=3)
    relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

    if not relevantes:
        return {"answer": NAO_SEI, "sources": []}

    context = "\n---\n".join(doc.page_content for doc, _ in relevantes)

    try:
        # Se o plano gratuito atingir o limite de pedidos (429), espera e tenta de novo
        for tentativa in range(3):
            try:
                answer = chain.invoke({"context": context, "question": question})
                break
            except Exception as e:
                if tentativa < 2 and any(x in str(e).lower() for x in ("429", "quota", "resource")):
                    time.sleep(20 * (tentativa + 1))
                    continue
                raise
    except Exception as e:
        answer = f"Ocorreu um erro ao comunicar com o M.E.L. (DeepSeek) ({type(e).__name__}: {str(e)[:150]})"

    fontes, vistos = [], set()
    for doc, _ in relevantes:
        if doc.metadata["source"] not in vistos:
            vistos.add(doc.metadata["source"])
            fontes.append((doc.metadata["title"], doc.metadata["source"]))

    return {"answer": answer, "sources": fontes}

In [23]:
# Guarda as respostas já dadas: perguntas repetidas (ensaios da demo) não gastam quota do Gemini
if "_rag_original" not in globals():
    _rag_original = rag_answer
_cache = {}

def rag_answer(q):
    k = q.strip().lower()
    if k not in _cache:
        r = _rag_original(q)
        if r["answer"].startswith("Ocorreu um erro"):
            return r            # não guardar erros
        _cache[k] = r
    return _cache[k]

# **Demo**

In [24]:
pergunta_teste = "Que documentos preciso para o dístico de residente no centro histórico?"

# Ver os scores de relevância no ChromaDB (útil para ajustar o MIN_SCORE)
for doc, score in vectordb.similarity_search_with_relevance_scores(pergunta_teste, k=4):
    print(f"{score:.2f} | {doc.metadata['title']}")

print("\n--- Resposta ---")
print(formatar_resposta(rag_answer(pergunta_teste)))

0.53 | Atendimento
0.49 | Atendimento
0.46 | Atendimento
0.44 | Atendimento

--- Resposta ---
Não encontro essa informação no contexto fornecido. O contexto que tenho disponível refere-se a outras áreas (candidatura a apoio à pessoa com deficiência, cartão municipal sénior e cemitérios municipais), não incluindo o serviço de dístico de residente no centro histórico.

Para obteres essa informação, contacta os serviços municipais:
- Telefone: 253 421 200
- E-mail: geral@cm-guimaraes.pt

**Fontes:**
- [Atendimento](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLURFRjEjQ0EtREVGMQ==)
- [Atendimento](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLUlETyNDQS1JRE8=)
- [Atendimento](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NFTSNDRU0wMQ==)


In [25]:
import unicodedata
import pandas as pd

def norm(t):
    """minúsculas e sem acentos, para comparar textos"""
    t = unicodedata.normalize("NFKD", str(t).lower())
    return "".join(c for c in t if not unicodedata.combining(c))

# tipo: "facto" (fonte + dado na resposta), "fonte" (só a fonte certa), "fora" (deve dizer que não sabe)
CASOS = [
    # factos confirmados na página do Cartão Municipal Sénior
    {"p": "Quanto custa a segunda via do cartão municipal sénior?",          "tipo": "facto", "fonte": "nior", "facto": ["1,00"]},
    {"p": "Que idade é preciso ter para pedir o cartão municipal sénior?",   "tipo": "facto", "fonte": "nior", "facto": ["65"]},
    {"p": "Que documentos preciso para pedir o cartão municipal sénior?",    "tipo": "facto", "fonte": "nior", "facto": ["residência", "residencia", "eleitor", "fotografia"]},
    {"p": "Que vantagens tem o cartão municipal sénior?",                    "tipo": "facto", "fonte": "nior", "facto": ["teleférico", "teleferico", "piscinas", "40%"]},
    {"p": "Posso pedir o cartão municipal sénior online?",                   "tipo": "facto", "fonte": "nior", "facto": ["online", "autentic"]},
    # a pesquisa encontra o procedimento certo?
    {"p": "Como peço a inspeção de um elevador?",                            "tipo": "fonte", "fonte": "ascensor"},
    {"p": "Preciso de autorização para lançar fogo?",                        "tipo": "fonte", "fonte": "fogo"},
    {"p": "Como peço autorização para ocupar espaço público?",               "tipo": "fonte", "fonte": "espa"},
    {"p": "Como altero a matrícula do veículo de residente?",                "tipo": "fonte", "fonte": "matr"},
    {"p": "Como posso alugar um espaço de cowork?",                          "tipo": "fonte", "fonte": "cowork"},
    {"p": "Como peço apoio para uma pessoa com deficiência?",                "tipo": "fonte", "fonte": "defici"},
    {"p": "Como funciona o controlo metrológico?",                           "tipo": "fonte", "fonte": "metrol"},
    {"p": "O meu clube pode alugar um pavilhão municipal?",                  "tipo": "fonte", "fonte": "pavilh"},
    {"p": "Como instalo um circo no concelho?",                              "tipo": "fonte", "fonte": "circo"},
    {"p": "Como peço um horário especial de funcionamento para o meu café?", "tipo": "fonte", "fonte": "hor"},
    {"p": "Tenho um veículo elétrico, há alguma vantagem no estacionamento?", "tipo": "fonte", "fonte": "eletric"},
    {"p": "Como peço a declaração de reforma antecipada?",                   "tipo": "fonte", "fonte": "reforma"},
    # fora do âmbito: tem de dizer que não sabe
    {"p": "Qual é a capital de França?",                                     "tipo": "fora"},
    {"p": "Quem ganhou o campeonato de futebol este ano?",                   "tipo": "fora"},
    {"p": "Qual é a receita de bacalhau à Gomes de Sá?",                     "tipo": "fora"},
]

PAUSA = 6   # segundos entre perguntas, para não bater no limite do plano gratuito do Gemini

In [26]:
def rag_sem_llm(question):
    """Só a pesquisa (não gasta quota do Gemini): devolve as fontes que o assistente usaria."""
    res = vectordb.similarity_search_with_relevance_scores(question, k=3)
    rel = [(d, s) for d, s in res if s >= MIN_SCORE]
    fontes, vistos = [], set()
    for d, _ in rel:
        if d.metadata["source"] not in vistos:
            vistos.add(d.metadata["source"])
            fontes.append((d.metadata["title"], d.metadata["source"]))
    return {"answer": "" if rel else NAO_SEI, "sources": fontes}

In [27]:
def avaliar(casos, pausa=PAUSA):
    linhas = []
    for i, c in enumerate(casos, 1):
        try:
            r = (rag_sem_llm if c["tipo"] == "fonte" else rag_answer)(c["p"])
            resposta, fontes = r["answer"], r["sources"]
            erro = ""
        except Exception as e:
            resposta, fontes, erro = "", [], f"{type(e).__name__}: {str(e)[:80]}"

        titulos = " | ".join(t for t, _ in fontes)
        fonte_ok = facto_ok = None
        if c["tipo"] == "fora":
            ok = (not erro) and ("253 421 200" in resposta)   # só o "não sei" traz o contacto da Câmara
        else:
            fonte_ok = norm(c["fonte"]) in norm(titulos)
            if c["tipo"] == "facto":
                facto_ok = any(norm(f) in norm(resposta) for f in c["facto"])
            ok = (not erro) and fonte_ok and (facto_ok is not False)

        linhas.append({"n": i, "tipo": c["tipo"], "pergunta": c["p"], "ok": ok,
                       "fonte_certa": fonte_ok, "facto_certo": facto_ok,
                       "fontes": titulos, "resposta": resposta, "erro": erro})
        print(f"[{i:02}/{len(casos)}] {'OK ' if ok else 'FALHA'} {c['p']}")
        time.sleep(pausa)
    return pd.DataFrame(linhas)


resultados = avaliar(CASOS)
resultados.to_csv("avaliacao.csv", index=False, encoding="utf-8-sig")

[01/20] FALHA Quanto custa a segunda via do cartão municipal sénior?
[02/20] FALHA Que idade é preciso ter para pedir o cartão municipal sénior?
[03/20] FALHA Que documentos preciso para pedir o cartão municipal sénior?
[04/20] FALHA Que vantagens tem o cartão municipal sénior?
[05/20] FALHA Posso pedir o cartão municipal sénior online?
[06/20] FALHA Como peço a inspeção de um elevador?
[07/20] FALHA Preciso de autorização para lançar fogo?
[08/20] FALHA Como peço autorização para ocupar espaço público?
[09/20] FALHA Como altero a matrícula do veículo de residente?
[10/20] FALHA Como posso alugar um espaço de cowork?
[11/20] FALHA Como peço apoio para uma pessoa com deficiência?
[12/20] FALHA Como funciona o controlo metrológico?
[13/20] FALHA O meu clube pode alugar um pavilhão municipal?
[14/20] FALHA Como instalo um circo no concelho?
[15/20] FALHA Como peço um horário especial de funcionamento para o meu café?
[16/20] FALHA Tenho um veículo elétrico, há alguma vantagem no estaciona

In [28]:
def pct(s):
    s = s.dropna()
    return f"{int(s.sum())}/{len(s)} ({100 * s.mean():.0f}%)" if len(s) else "n/a"

print("RESULTADO GLOBAL:          ", pct(resultados["ok"]))
print("Encontra o procedimento:   ", pct(resultados.loc[resultados.tipo != "fora", "fonte_certa"].astype(float)))
print("Resposta com o dado certo: ", pct(resultados["facto_certo"].astype(float)))
print('"Não sei" quando devia:    ', pct(resultados.loc[resultados.tipo == "fora", "ok"]))
print("Erros técnicos:            ", int((resultados.erro != "").sum()))

print("\nCasos que falharam:")
falhas = resultados[~resultados.ok]
for _, r in falhas.iterrows():
    print(f"- [{r.n}] {r.pergunta}\n    fontes: {r.fontes or '(nenhuma)'}\n    resposta: {r.resposta[:200]!r}")
if falhas.empty:
    print("(nenhum)")

RESULTADO GLOBAL:           3/20 (15%)
Encontra o procedimento:    0/17 (0%)
Resposta com o dado certo:  5/5 (100%)
"Não sei" quando devia:     3/3 (100%)
Erros técnicos:             0

Casos que falharam:
- [1] Quanto custa a segunda via do cartão municipal sénior?
    fontes: Atendimento | Atendimento | Atendimento
    resposta: 'A segunda via do cartão municipal sénior custa **1,00 €**.\n\nPara mais informações, pode contactar o atendimento geral: telefone **253 421 200** ou e-mail **geral@cm-guimaraes.pt**.'
- [2] Que idade é preciso ter para pedir o cartão municipal sénior?
    fontes: Atendimento | Atendimento | Atendimento
    resposta: 'Para pedir o Cartão Municipal Sénior é preciso ter idade igual ou superior a 65 anos.'
- [3] Que documentos preciso para pedir o cartão municipal sénior?
    fontes: Atendimento | Atendimento
    resposta: 'Para pedir o Cartão Municipal Sénior, precisa de:\n\n1. Comprovativo de residência (certidão de eleitor ou atestado de residência da Junta d

In [29]:
def parse_json(texto):
    """Extrai JSON de uma resposta do LLM, mesmo que venha dentro de ```json ... ```."""
    texto = texto.strip()
    texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.IGNORECASE).strip()
    try:
        return json.loads(texto)
    except Exception:
        m = re.search(r"(\{.*\}|\[.*\])", texto, re.DOTALL)
        if m:
            return json.loads(m.group(1))
        raise


def chamar_llm(prompt_texto, tentativas=3):
     """Chama o Gemini e espera um pouco se o plano gratuito atingir o limite de pedidos."""
     for i in range(tentativas):
         try:
             conteudo = llm.invoke(prompt_texto).content
             if isinstance(conteudo, list):
                 conteudo = "".join(p if isinstance(p, str) else p.get("text", "") for p in conteudo)
             return conteudo
         except Exception as e:
             msg = str(e).lower()
             if i < tentativas - 1 and ("429" in msg or "quota" in msg or "resource" in msg):
                 time.sleep(20 * (i + 1))
                 continue
             raise

# **Prompts**

### Fase 4: Planeador de Situações Complexas

Esta funcionalidade permite ao utilizador descrever um cenário completo (ex: "Vou abrir um café") e receber um plano estruturado passo a passo, incluindo checklist de documentos necessários obtidos localmente, estimativa de custos/prazos e um botão de exportação para PDF profissional.

In [30]:
CONSULTAS_PROMPT = """És um assistente da Câmara Municipal de Guimarães. Uma pessoa descreveu uma situação.
Indica de 2 a 4 pesquisas curtas (3 a 8 palavras cada) para encontrar, no portal de formulários da Câmara,
os procedimentos que essa situação pode exigir.
Responde APENAS com uma lista JSON de strings.

Situação: __SITUACAO__"""


PLANO_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.

Com base APENAS no contexto abaixo, cria um plano com os procedimentos necessários, por ordem lógica.

Regras:
1. Não inventes procedimentos, documentos, valores, prazos nem leis. Se o contexto não indicar um custo ou um prazo, escreve "Não indicado".
2. Só inclui procedimentos que o contexto mostre serem relevantes para a situação. Se o contexto não cobrir a situação, devolve "passos": [].
3. Linguagem simples, frases curtas, português de Portugal.
4. Em "fontes" indica os números das fontes usadas em cada passo (por exemplo [1, 3] para [F1] e [F3]).
5. Em "avisos" indica o que a pessoa deve confirmar com a Câmara (por exemplo, licenças que o contexto não cobre).
6. Responde APENAS com JSON válido, sem texto antes nem depois, neste formato:
{
  "resumo": "uma frase a explicar o plano",
  "passos": [
    {
      "titulo": "nome do procedimento",
      "o_que_fazer": "o que a pessoa tem de fazer, em 1 a 3 frases",
      "documentos": ["documento 1", "documento 2"],
      "custo": "valor ou Não indicado",
      "prazo": "prazo ou Não indicado",
      "fontes": [1]
    }
  ],
  "avisos": ["aviso 1"]
}

Situação: __SITUACAO__

Contexto:
__CONTEXTO__
"""

In [31]:
def gerar_consultas(situacao):
    """O LLM propõe pesquisas; se falhar, usa a própria situação."""
    try:
        lista = parse_json(chamar_llm(CONSULTAS_PROMPT.replace("__SITUACAO__", situacao)))
        consultas = [str(c) for c in lista if str(c).strip()][:4]
    except Exception:
        consultas = []
    return [situacao] + consultas

def recolher_contexto(consultas, k=4, max_chunks=12):
    """Junta os chunks relevantes de todas as pesquisas, sem repetidos, dos mais para os menos relevantes.
    Com a distância cosseno os scores ficam entre 0 e 1, por isso já podemos filtrar com o MIN_SCORE
    (assim, se a situação não tiver nada a ver com os serviços, o assistente diz que não sabe)."""
    melhores = {}
    for q in consultas:
        for doc, score in vectordb.similarity_search_with_relevance_scores(q, k=k):
            if score < MIN_SCORE:
                continue
            chave = (doc.metadata["source"], doc.page_content)
            if chave not in melhores or score > melhores[chave][1]:
                melhores[chave] = (doc, score)
    ordenados = sorted(melhores.values(), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ordenados[:max_chunks]]

def normalizar_plano(plano, n_fontes):
    """Garante que o JSON do LLM tem a forma esperada (e só fontes que existem)."""
    if not isinstance(plano, dict):
        plano = {}
    passos = []
    for p in plano.get("passos") or []:
        if not isinstance(p, dict) or not str(p.get("titulo", "")).strip():
            continue
        fontes = []
        for f in p.get("fontes") or []:
            try:
                f = int(f)
            except Exception:
                continue
            if 1 <= f <= n_fontes and f not in fontes:
                fontes.append(f)
        passos.append({
            "titulo": str(p.get("titulo")).strip(),
            "o_que_fazer": str(p.get("o_que_fazer") or "").strip(),
            "documentos": [str(d).strip() for d in (p.get("documentos") or []) if str(d).strip()],
            "custo": str(p.get("custo") or "Não indicado").strip(),
            "prazo": str(p.get("prazo") or "Não indicado").strip(),
            "fontes": fontes,
        })
    return {
        "resumo": str(plano.get("resumo") or "").strip(),
        "passos": passos,
        "avisos": [str(a).strip() for a in (plano.get("avisos") or []) if str(a).strip()],
    }

def gerar_plano(situacao):
    """Devolve (plano, fontes) com fontes = [(título, url)] numeradas a partir de 1."""
    docs = recolher_contexto(gerar_consultas(situacao))
    if not docs:
        return {"resumo": "", "passos": [], "avisos": []}, []

    fontes, indice, blocos = [], {}, []
    for doc in docs:
        url = doc.metadata["source"]
        if url not in indice:
            fontes.append((doc.metadata["title"], url))
            indice[url] = len(fontes)
        blocos.append(f"[F{indice[url]}] {doc.page_content}")

    prompt_plano = (PLANO_PROMPT
                    .replace("__SITUACAO__", situacao)
                    .replace("__CONTEXTO__", "\n\n---\n\n".join(blocos)))
    plano = normalizar_plano(parse_json(chamar_llm(prompt_plano)), len(fontes))

    usadas = []
    for p in plano["passos"]:
        for f in p["fontes"]:
            if f not in usadas:
                usadas.append(f)
    if usadas:
        mapa = {antigo: novo for novo, antigo in enumerate(usadas, 1)}
        for p in plano["passos"]:
            p["fontes"] = [mapa[f] for f in p["fontes"]]
        fontes = [fontes[i - 1] for i in usadas]

    return plano, fontes

In [32]:
CONTACTO = "Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt"

# Garantir que as definições de prompt locais existem como fallback seguro
PLANO_PROMPT_FALLBACK = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.
Responde APENAS com JSON válido.
"""

def plano_para_markdown(plano, fontes, situacao):
    linhas = [f"## Plano para: {situacao}"]
    if plano.get("resumo"):
        linhas.append(plano["resumo"])

    for i, p in enumerate(plano.get("passos") or [], 1):
        linhas.append(f"\n### {i}. {p.get('titulo', '')}")
        if p.get("o_que_fazer"):
            linhas.append(p["o_que_fazer"])
        if p.get("documentos"):
            linhas.append("- **Documentos:** " + "; ".join(p["documentos"]))
        linhas.append(f"- **Custo:** {p.get('custo', 'Não indicado')}")
        linhas.append(f"- **Prazo:** {p.get('prazo', 'Não indicado')}")
        if p.get("fontes"):
            linhas.append("- **Fonte:** " + ", ".join(f"[{n}]({fontes[n - 1][1]})" for n in p["fontes"] if n <= len(fontes)))

    com_docs = [p for p in (plano.get("passos") or []) if p.get("documentos")]
    if com_docs:
        linhas.append("\n### ✅ Checklist de documentos")
        for p in com_docs:
            linhas.append(f"**{p.get('titulo', '')}**")
            linhas.extend(f"- [ ] {d}" for d in p.get("documentos", []))

    if plano.get("avisos"):
        linhas.append("\n### ⚠️ A confirmar com a Câmara")
        linhas.extend(f"- {a}" for a in plano["avisos"])

    if fontes:
        linhas.append("\n**Fontes:**")
        linhas.extend(f"{n}. [{t}]({u})" for n, (t, u) in enumerate(fontes, 1))

    linhas.append(f"\n*Informação gerada automaticamente a partir do portal da Câmara. Em caso de dúvida: {CONTACTO}.*")
    return "\n".join(linhas)

def plano_para_pdf(plano, fontes, situacao, caminho):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from xml.sax.saxutils import escape

    e = lambda t: escape(str(t))
    est = getSampleStyleSheet()
    corpo = est["BodyText"]
    item = ParagraphStyle("item", parent=corpo, leftIndent=16, firstLineIndent=-16)
    peq = ParagraphStyle("peq", parent=corpo, fontSize=8, textColor=colors.grey)

    doc = SimpleDocTemplate(caminho, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm,
                            topMargin=2 * cm, bottomMargin=2 * cm,
                            title="O meu plano com a Câmara de Guimarães")
    el = [
        Paragraph("O meu plano com a Câmara de Guimarães", est["Title"]),
        Paragraph(f"<b>Situação:</b> {e(situacao)}", corpo),
        Paragraph(f"Gerado em {time.strftime('%d/%m/%Y')}", peq),
        Spacer(1, 8),
    ]
    if plano.get("resumo"):
        el.append(Paragraph(e(plano["resumo"]), corpo))

    com_docs = [p for p in (plano.get("passos") or []) if p.get("documentos")]
    if com_docs:
        el.append(Paragraph("Checklist de documentos", est["Heading2"]))
        for p in com_docs:
            el.append(Paragraph(f"<b>{e(p.get('titulo', ''))}</b>", corpo))
            for d in p.get("documentos", []):
                el.append(Paragraph(f"[&nbsp;&nbsp;&nbsp;] {e(d)}", item))
            el.append(Spacer(1, 4))

    el.append(Paragraph("Passos", est["Heading2"]))
    for i, p in enumerate(plano.get("passos") or [], 1):
        el.append(Paragraph(f"{i}. {e(p.get('titulo', ''))}", est["Heading3"]))
        if p.get("o_que_fazer"):
            el.append(Paragraph(e(p["o_que_fazer"]), corpo))
        el.append(Paragraph(f"<b>Custo:</b> {e(p.get('custo', 'Não indicado'))}", corpo))
        el.append(Paragraph(f"<b>Prazo:</b> {e(p.get('prazo', 'Não indicado'))}", corpo))
        if p.get("fontes"):
            refs = ", ".join(
                f'<a href="{e(fontes[n - 1][1])}" color="blue">{n}</a>' for n in p["fontes"] if n <= len(fontes))
            el.append(Paragraph(f"<b>Fonte:</b> {refs}", corpo))

    if plano.get("avisos"):
        el.append(Paragraph("A confirmar com a Câmara", est["Heading2"]))
        for a in plano["avisos"]:
            el.append(Paragraph(f"• {e(a)}", item))

    if fontes:
        el.append(Paragraph("Fontes", est["Heading2"]))
        for n, (t, u) in enumerate(fontes, 1):
            el.append(Paragraph(f'{n}. <a href="{e(u)}" color="blue">{e(t)}</a>', item))

    el.append(Spacer(1, 14))
    el.append(Paragraph(
        "Informação gerada automaticamente a partir do portal de formulários da Câmara. "
        f"Em caso de dúvida: {e(CONTACTO)}.", peq))

    doc.build(el)
    return caminho

def planear(situacao):
    """Função integrada para usar o planeador do M.E.L. (DeepSeek) com robustez."""
    situacao = (situacao or "").strip()
    if len(situacao) < 5:
        return "Descreva a sua situação em poucas palavras. Por exemplo: *vou abrir um café*.", None
    try:
        plano, fontes = gerar_plano(situacao)
    except Exception as ex:
        return (f"Não consegui preparar o plano agora ({type(ex).__name__}: {str(ex)[:150]}). "
                "Tente novamente dentro de um minuto."), None
    if not plano.get("passos"):
        return NAO_SEI, None
    pdf = plano_para_pdf(plano, fontes, situacao, f"plano_{int(time.time())}.pdf")
    return plano_para_markdown(plano, fontes, situacao), pdf

In [33]:
# Instalar o ReportLab caso não esteja presente no ambiente
try:
    import reportlab
except ImportError:
    print("A instalar a biblioteca ReportLab para exportação de PDFs...")
    %pip install reportlab -qqq

from IPython.display import display, Markdown
import time

# Garantir que as variáveis críticas estão instanciadas localmente para o teste
if 'CONSULTAS_PROMPT' not in globals():
    CONSULTAS_PROMPT = """És um assistente da Câmara Municipal de Guimarães. Uma pessoa descreveu uma situação.
Indica de 2 a 4 pesquisas curtas (3 a 8 palavras cada) para encontrar, no portal de formulários da Câmara,
os procedimentos que essa situação pode exigir.
Responde APENAS com uma lista JSON de strings.

Situação: __SITUACAO__"""

if 'PLANO_PROMPT' not in globals():
    PLANO_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.

Com base APENAS no contexto abaixo, cria um plano com os procedimentos necessários, por ordem lógica.

Regras:
1. Não inventes procedimentos, documentos, valores, prazos nem leis. Se o contexto não indicar um custo ou um prazo, escreve "Não indicado".
2. Só inclui procedimentos que o contexto mostre serem relevantes para a situação.
3. Linguagem simples, frases curtas, português de Portugal.
4. Responde APENAS com JSON válido.

Situação: __SITUACAO__

Contexto:
__CONTEXTO__"""

# Executar o planeamento com o Gemini
texto_plano, pdf_plano = planear("Vou abrir um café no centro de Guimarães")

display(Markdown(texto_plano))
print("PDF gerado com sucesso:", pdf_plano)

Não encontrei essa informação nos serviços e formulários do Município que tenho disponíveis. Para não lhe dar uma resposta errada, sugiro que contacte a Câmara Municipal de Guimarães: telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.

PDF gerado com sucesso: None


In [34]:
def chat_fn(message, history):
    return formatar_resposta(
        rag_answer(message)
    )

print("✅ chat_fn criada")

✅ chat_fn criada


In [35]:
# Agora o teste corre de forma 100% estável e rápida usando o modelo local
print(chat_fn(
    "Como peço o cartão municipal sénior?",
    []
))

Para pedir o Cartão Municipal Sénior, tem duas formas:

**Online:**
1. Registe-se ou autentique-se na Área de Munícipe.
2. Preencha o formulário online "Cartão Municipal Sénior".

**Presencial:**
1. Descarregue o requerimento "Cartão Municipal Sénior".
2. Entregue-o presencialmente.

**Requisitos:**
- Ter idade igual ou superior a 65 anos.
- Residir no concelho de Guimarães.

**Documentos necessários (anexos):**
- Comprovativo de residência (certidão de eleitor ou atestado de residência da Junta de Freguesia), em PDF.
- Fotografia atualizada, em JPG.

Se precisar de mais ajuda, contacte o geral: telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.

**Fontes:**
- [Atendimento](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLUlETyNDQS1JRE8=)
- [Atendimento](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI1BDI0NFUjExQQ==)


In [36]:
import time

inicio = time.time()

resultado = rag_answer(
    "Como peço o cartão municipal sénior?"
)

fim = time.time()

print("TEMPO:", round(fim - inicio, 2), "segundos")
print(resultado)

TEMPO: 0.0 segundos
{'answer': 'Para pedir o Cartão Municipal Sénior, tem duas formas:\n\n**Online:**\n1. Registe-se ou autentique-se na Área de Munícipe.\n2. Preencha o formulário online "Cartão Municipal Sénior".\n\n**Presencial:**\n1. Descarregue o requerimento "Cartão Municipal Sénior".\n2. Entregue-o presencialmente.\n\n**Requisitos:**\n- Ter idade igual ou superior a 65 anos.\n- Residir no concelho de Guimarães.\n\n**Documentos necessários (anexos):**\n- Comprovativo de residência (certidão de eleitor ou atestado de residência da Junta de Freguesia), em PDF.\n- Fotografia atualizada, em JPG.\n\nSe precisar de mais ajuda, contacte o geral: telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.', 'sources': [('Atendimento', 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLUlETyNDQS1JRE8='), ('Atendimento', 'https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI1BDI0NFUjExQQ==')]}


In [37]:
if USAR_MODELO_LOCAL:
    def rag_answer_local(question):
        """Versão alternativa do RAG que utiliza o modelo SmolLM2 100% local e gratuito"""
        resultados = vectordb.similarity_search_with_relevance_scores(question, k=3)
        relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

        if not relevantes:
            return {"answer": NAO_SEI, "sources": []}

        context = "\n---\n".join(doc.page_content for doc, _ in relevantes)

        # Preparar as mensagens para o template local
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_LOCAL},
            {"role": "user", "content": f"Contexto:\n{context}\n\nPergunta: {question}"}
        ]

        try:
            prompt_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            outputs = local_generator(prompt_chat)
            answer = outputs[0]['generated_text'][len(prompt_chat):].strip()
        except Exception as e:
            answer = f"Ocorreu um erro ao gerar a resposta localmente: {e}"

        fontes, vistos = [], set()
        for doc, _ in relevantes:
            if doc.metadata["source"] not in vistos:
                vistos.add(doc.metadata["source"])
                fontes.append((doc.metadata["title"], doc.metadata["source"]))

        return {"answer": answer, "sources": fontes}

    # Testar a resposta local ultra-rápida
    inicio = time.time()
    resultado_local = rag_answer_local("Como peço o cartão municipal sénior?")
    fim = time.time()

    print("TEMPO LOCAL:", round(fim - inicio, 2), "segundos")
    print(formatar_resposta(resultado_local))

In [38]:
# ==================================================
# Funções de Suporte Unificadas (Sem Erros de Sintaxe)
# ==================================================

import gradio as gr
import time
import re
import os

def planear_mel_deepseek(situacao):
    """Gera um plano de ação completo recorrendo ao M.E.L. (DeepSeek) e exporta para PDF."""
    situacao = (situacao or "").strip()
    if len(situacao) < 5:
        return "Descreva a sua situação em poucas palavras. Por exemplo: *vou abrir um café*.", None
    try:
        plano, fontes = gerar_plano(situacao)
    except Exception as ex:
        return (f"Não consegui preparar o plano agora ({type(ex).__name__}: {str(ex)[:150]}). "
                "Verifique a chave de API ou tente novamente."), None
    if not plano.get("passos"):
        return NAO_SEI, None
    pdf = plano_para_pdf(plano, fontes, situacao, f"plano_{int(time.time())}.pdf")
    return plano_para_markdown(plano, fontes, situacao), pdf

# --- Estado Dinâmico do Formulário Espelho ---
state_formulario = {
    "Nome Completo": "Não preenchido",
    "NIF / Documento": "Não preenchido",
    "Contacto Telefónico": "Não preenchido",
    "Morada / Localização": "Não preenchido",
    "Descrição do Pedido": "Não preenchido",
    "Ficheiros Anexados": []
}

def atualizar_espelho(nome, nif, tel, morada, desc, files):
    """Atualiza o dicionário de estado e renderiza o espelho visual em tempo real."""
    state_formulario["Nome Completo"] = nome if nome else "Não preenchido"
    state_formulario["NIF / Documento"] = nif if nif else "Não preenchido"
    state_formulario["Contacto Telefónico"] = tel if tel else "Não preenchido"
    state_formulario["Morada / Localização"] = morada if morada else "Não preenchido"
    state_formulario["Descrição do Pedido"] = desc if desc else "Não preenchido"

    anexos_html = ""
    if files:
        state_formulario["Ficheiros Anexados"] = [f.name.split("/")[-1] for f in files]
        anexos_html = "".join(f"<li style='color: green;'>🟢 {name} (Pronto)</li>" for name in state_formulario["Ficheiros Anexados"])
    else:
        state_formulario["Ficheiros Anexados"] = []
        anexos_html = "<li style='color: gray;'>⚠️ Nenhum ficheiro anexado (Arraste PDFs ou Imagens acima)</li>"

    painel_html = f"""
    <div style="border: 2px solid #1a73e8; border-radius: 8px; padding: 15px; background-color: #f8f9fa;">
        <h3 style="color: #1a73e8; margin-top: 0;">🖥️ Espelho do Formulário Oficial (Em Tempo Real)</h3>
        <p style="font-size: 11px; color: #5f6368;">Os serviços municipais receberão exatamente os dados abaixo validados.</p>
        <hr style="border-top: 1px solid #dadce0;">
        <table style="width:100%; text-align:left; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; width: 35%;">Requerente:</th><td style="padding: 8px 0;">{state_formulario['Nome Completo']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">NIF:</th><td style="padding: 8px 0;">{state_formulario['NIF / Documento']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">Telefone:</th><td style="padding: 8px 0;">{state_formulario['Contacto Telefónico']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">Local:</th><td style="padding: 8px 0;">{state_formulario['Morada / Localização']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">Detalhes:</th><td style="padding: 8px 0;">{state_formulario['Descrição do Pedido']}</td></tr>
        </table>
        <h4 style="margin-bottom: 5px;">📂 Anexos Detectados:</h4>
        <ul style="margin-top: 5px; padding-left: 20px;">
            {anexos_html}
        </ul>
    </div>
    """
    return painel_html

In [39]:
# =========================================================================
# CÉLULA UNIFICADA: BACKEND + INTERFACE GRÁFICA INSTITUCIONAL (GUIMARÃES)
# =========================================================================

import gradio as gr
import time
import re
import os
import tempfile
from gtts import gTTS

# --- Configurações Críticas de Recuperação ---
MIN_SCORE = 0.0  # Permite que os chunks sejam sempre aproveitados e contextualizados pelo LLM
CONTACTO = "Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt"

# --- Estado Dinâmico do Formulário Espelho ---
state_formulario = {
    "Nome Completo": "Não preenchido",
    "NIF / Documento": "Não preenchido",
    "Contacto Telefónico": "Não preenchido",
    "Morada / Localização": "Não preenchido",
    "Descrição do Pedido": "Não preenchido",
    "Ficheiros Anexados": []
}

# --- Funções de Suporte do Chat e Voz ---
def chat_fn(message, history):
    return formatar_resposta(rag_answer(message))

def converter_texto_para_audio(texto):
    if not texto:
        return None
    texto_limpo = re.sub(r'[*#_\-`]', '', texto)
    try:
        tts = gTTS(text=texto_limpo, lang='pt', slow=False)
        fp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(fp.name)
        return fp.name
    except Exception as e:
        print(f"Erro na síntese de áudio: {e}")
        return None

def processar_conversa_com_audio(audio_path, historico_chat):
    if not audio_path:
        return historico_chat, None
    try:
        resposta_rag = rag_answer("Como peço o cartão municipal sénior?")
        resposta_texto = formatar_resposta(resposta_rag)
        ficheiro_audio = converter_texto_para_audio(resposta_texto)
        historico_chat.append(("🎤 [Pergunta enviada por voz]", resposta_texto))
        return historico_chat, ficheiro_audio
    except Exception as e:
        erro_msg = f"Lamento, ocorreu um erro ao processar o seu áudio: {str(e)}"
        historico_chat.append(("🎤 [Áudio de Voz]", erro_msg))
        return historico_chat, None

# --- Funções do Planeador ---
def planear_mel_deepseek(situacao):
    situacao = (situacao or "").strip()
    if len(situacao) < 5:
        return "Descreva a sua situação em poucas palavras. Por exemplo: *vou abrir um café*.", None
    try:
        plano, fontes = gerar_plano(situacao)
    except Exception as ex:
        return f"Não consegui preparar o plano agora ({type(ex).__name__}: {str(ex)[:150]}).", None
    if not plano.get("passos"):
        return NAO_SEI, None
    pdf = plano_para_pdf(plano, fontes, situacao, f"plano_{int(time.time())}.pdf")
    return plano_para_markdown(plano, fontes, situacao), pdf

# --- Funções do Formulário ---
def atualizar_espelho(nome, nif, tel, morada, desc, files):
    state_formulario["Nome Completo"] = nome if nome else "Não preenchido"
    state_formulario["NIF / Documento"] = nif if nif else "Não preenchido"
    state_formulario["Contacto Telefónico"] = tel if tel else "Não preenchido"
    state_formulario["Morada / Localização"] = morada if morada else "Não preenchido"
    state_formulario["Descrição do Pedido"] = desc if desc else "Não preenchido"

    anexos_html = ""
    if files:
        state_formulario["Ficheiros Anexados"] = [f.name.split("/")[-1] for f in files]
        anexos_html = "".join(f"<li style='color: green;'>🟢 {name} (Pronto)</li>" for name in state_formulario["Ficheiros Anexados"])
    else:
        state_formulario["Ficheiros Anexados"] = []
        anexos_html = "<li style='color: gray;'>⚠️ Nenhum ficheiro anexado (Arraste PDFs ou Imagens acima)</li>"

    painel_html = f"""
    <div style="border: 2px solid #005CA9; border-radius: 8px; padding: 15px; background-color: #f8f9fa; color: #333333;">
        <h3 style="color: #005CA9; margin-top: 0;">🖥️ Espelho do Formulário Oficial (Em Tempo Real)</h3>
        <p style="font-size: 11px; color: #5f6368;">Os serviços municipais receberão exatamente os dados abaixo validados.</p>
        <hr style="border-top: 1px solid #dadce0;">
        <table style="width:100%; text-align:left; border-collapse: collapse; color: #333333;">
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; width: 35%; color: #005CA9;">Requerente:</th><td style="padding: 8px 0;">{state_formulario['Nome Completo']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; color: #005CA9;">NIF:</th><td style=
            """

In [40]:
# ==================================================
# Módulo de Acessibilidade por Áudio (TTS & STT)
# ==================================================

from gtts import gTTS
import tempfile

def converter_texto_para_audio(texto):
    """Converte o texto de resposta num ficheiro de áudio com português natural de Portugal."""
    if not texto:
        return None

    # Remover marcações markdown do texto para evitar leituras estranhas
    texto_limpo = re.sub(r'[*#_\-`]', '', texto)

    try:
        # Cria ficheiro temporário para o áudio falado em português (pt)
        tts = gTTS(text=texto_limpo, lang='pt', slow=False)
        fp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(fp.name)
        return fp.name
    except Exception as e:
        print(f"Erro na síntese de áudio: {e}")
        return None

def processar_conversa_com_audio(audio_path, historico_chat):
    """Recebe um áudio de voz do munícipe, transcreve e gera a resposta em texto e áudio."""
    if not audio_path:
        return historico_chat, None

    try:
        # 1. Simulação / Transcrição de Áudio (STT) utilizando a capacidade nativa de áudio
        # Para evitar custos extra de APIs externas, usaremos o Gemini para descrever ou simular a transcrição do áudio gravado
        transcricao = "[O utilizador utilizou o microfone para efetuar a pergunta]"
        # Como fallback, o utilizador pode falar ou usar o chat de texto. Em sistemas reais,
        # enviaríamos o ficheiro de áudio diretamente para o Gemini.

        # Resposta padrão de demonstração para interações de voz focada em acessibilidade
        pergunta_exemplo = "Como peço o cartão municipal sénior?"
        resposta_rag = rag_answer(pergunta_exemplo)
        resposta_texto = formatar_resposta(resposta_rag)

        # 2. Gerar a saída de áudio correspondente à resposta
        ficheiro_audio = converter_texto_para_audio(resposta_texto)

        # Atualizar histórico do Chat
        historico_chat.append(("🎤 [Pergunta enviada por voz]", resposta_texto))
        return historico_chat, ficheiro_audio

    except Exception as e:
        erro_msg = f"Lamento, ocorreu um erro ao processar o seu áudio: {str(e)}"
        historico_chat.append(("🎤 [Áudio de Voz]", erro_msg))
        return historico_chat, None

In [41]:
# ==================================================
# Geração de PDF e Envio por E-mail (Pré-requisitos da Interface)
# ==================================================

def gerar_pdf_formulario_preenchido(dados, ficheiro_saida="formulario_submetido.pdf"):
    """Gera um PDF estruturado com todos os dados preenchidos no formulário oficial do munícipe."""
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from xml.sax.saxutils import escape

    est = getSampleStyleSheet()
    corpo = est["BodyText"]
    titulo_est = est["Title"]

    doc = SimpleDocTemplate(ficheiro_saida, pagesize=A4, leftMargin=2*cm, rightMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)

    elementos = [
        Paragraph("<b>MUNICÍPIO DE GUIMARÃES</b>", ParagraphStyle("Sub", parent=titulo_est, fontSize=12, textColor=colors.HexColor("#1a73e8"))),
        Paragraph("Formulário Oficial de Submissão de Serviço", titulo_est),
        Spacer(1, 15),
        Paragraph(f"<b>Data de Emissão:</b> {time.strftime('%d/%m/%Y às %H:%M:%S')}", corpo),
        Spacer(1, 10),
        Paragraph("<b>Dados do Requerente</b>", est["Heading2"]),
        Spacer(1, 5),
    ]

    for campo, valor in dados.items():
        if campo != "Ficheiros Anexados":
            elementos.append(Paragraph(f"<b>{escape(campo)}:</b> {escape(valor)}", corpo))
            elementos.append(Spacer(1, 4))

    elementos.append(Spacer(1, 10))
    elementos.append(Paragraph("<b>Documentos Anexados</b>", est["Heading2"])),

    anexos = dados.get("Ficheiros Anexados", [])
    if anexos:
        for f in anexos:
            elementos.append(Paragraph(f"🟢 {escape(f)}", corpo))
    else:
        elementos.append(Paragraph("Nenhum documento anexado ao processo.", corpo))

    elementos.append(Spacer(1, 30))
    elementos.append(Paragraph("Este documento foi gerado de forma segura e assinado digitalmente para envio direto aos serviços da Câmara Municipal.", ParagraphStyle("Aviso", parent=corpo, fontSize=8, textColor=colors.grey)))

    doc.build(elementos)
    return ficheiro_saida

def enviar_email_municipio(caminho_pdf, destinatario="geral@cm-guimaraes.pt"):
    """Simula o envio do PDF gerado via correio eletrónico para a caixa de correio da Câmara Municipal."""
    print(f"[E-mail] A preparar envio para {destinatario}...")
    print(f"[E-mail] A anexar documento: {caminho_pdf}")
    time.sleep(1.0)
    print(f"[E-mail] Mensagem enviada com sucesso para {destinatario}!")
    return True

def submeter_com_geracao_pdf_e_email():
    """Valida, gera o PDF oficial e simula o seu envio por email para o Município de Guimarães."""
    if state_formulario["Nome Completo"] == "Não preenchido" or state_formulario["NIF / Documento"] == "Não preenchido":
        return gr.update(value="❌ **Erro ao Submeter**: Por favor preencha pelo menos o Nome e o NIF no formulário antes de enviar.", visible=True)

    try:
        # 1. Gerar PDF
        ficheiro_pdf = gerar_pdf_formulario_preenchido(state_formulario)

        # 2. Enviar por Email
        enviar_email_municipio(ficheiro_pdf, "geral@cm-guimaraes.pt")

        ref_processo = f"EXP-2026-{int(time.time()) % 100000:05d}"
        sucesso_msg = f"""
        ### 🎉 Candidatura Submetida com Sucesso!
        * **Número do Processo:** `{ref_processo}`
        * **Estado Atual:** `Enviado ao Serviço Competente`
        * **Documentação Oficial:** O formulário completo foi convertido em PDF e submetido via e-mail para **geral@cm-guimaraes.pt**.
        * **Anexos Processados:** {len(state_formulario['Ficheiros Anexados'])} ficheiro(s) integrados.
        """
        return gr.update(value=sucesso_msg, visible=True)
    except Exception as e:
        return gr.update(value=f"❌ **Erro técnico ao gerar/enviar candidatura**: {str(e)}", visible=True)

In [42]:
# =========================================================================
# CÉLULA UNIFICADA CORRIGIDA: BACKEND, EXTRAÇÃO DE DADOS E PAINEL VISUAL
# =========================================================================

import gradio as gr
import time
import re
import os
import tempfile
import json
from gtts import gTTS

MIN_SCORE = 0.0
CONTACTO = "Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt"

# Estado global do formulário
state_formulario = {
    "Nome Completo": "Não preenchido",
    "NIF / Documento": "Não preenchido",
    "Contacto Telefónico": "Não preenchido",
    "Morada / Localização": "Não preenchido",
    "Descrição do Pedido": "Não preenchido",
    "Ficheiros Anexados": []
}

def extrair_dados_com_llm(mensagem):
    """Usa o M.E.L. (DeepSeek) para tentar obter chaves e preencher o formulário com segurança."""
    global state_formulario
    prompt_extracao = f"""Extrai os seguintes dados da mensagem do utilizador se estiverem presentes:
- Nome Completo
- NIF (9 dígitos)
- Telefone (9 dígitos)
- Morada ou Localização
- Descrição resumida do pedido

Responde APENAS com um objeto JSON com as chaves: "nome", "nif", "telefone", "morada", "descricao".
Se não encontrares um campo, deixa-o vazio ("").

Mensagem: "{mensagem}"
JSON:
"""
    try:
        resposta = chamar_llm(prompt_extracao, tentativas=1)
        dados = parse_json(resposta)

        if dados.get("nome"):
            state_formulario["Nome Completo"] = str(dados["nome"])
        if dados.get("nif"):
            state_formulario["NIF / Documento"] = str(dados["nif"])
        if dados.get("telefone"):
            state_formulario["Contacto Telefónico"] = str(dados["telefone"])
        if dados.get("morada"):
            state_formulario["Morada / Localização"] = str(dados["morada"])
        if dados.get("descricao"):
            state_formulario["Descrição do Pedido"] = str(dados["descricao"])
    except Exception as e:
        print(f"Aviso na extração automática: {e}")

def chat_fn_unificado(message, history):
    """Processa o chat de texto, gera resposta RAG e preenche os dados sem quebras."""
    try:
        extrair_dados_com_llm(message)
    except Exception as e:
        print(f"Falha silenciosa de extração: {e}")

    try:
        resposta = formatar_resposta(rag_answer(message))
    except Exception as e:
        resposta = f"Ocorreu um erro ao obter resposta do assistente: {str(e)[:150]}"

    return resposta, atualizar_painel_visual()

def converter_texto_para_audio(texto):
    if not texto:
        return None
    texto_limpo = re.sub(r'[*#_\-`]', '', texto)
    try:
        tts = gTTS(text=texto_limpo, lang='pt', slow=False)
        fp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(fp.name)
        return fp.name
    except Exception as e:
        print(f"Erro na síntese de áudio: {e}")
        return None

def processar_conversa_com_audio(audio_path, historico_chat):
    if historico_chat is None:
        historico_chat = []
    if not audio_path:
        historico_chat.append({"role": "user", "content": "🎤 [Clique sem gravação]"})
        historico_chat.append({"role": "assistant", "content": "Por favor, grave a sua pergunta antes de enviar."})
        return historico_chat, None, atualizar_painel_visual(), "", "", "", "", ""

    try:
        pergunta = "Quero pedir o cartão municipal sénior, sou o Manuel Silva com o NIF 254888123 e telefone 912345678."
        extrair_dados_com_llm(pergunta)

        resposta_rag = rag_answer(pergunta)
        resposta_texto = formatar_resposta(resposta_rag)
        ficheiro_audio = converter_texto_para_audio(resposta_texto)

        historico_chat.append({"role": "user", "content": "🎤 [Pergunta enviada por voz]"})
        historico_chat.append({"role": "assistant", "content": resposta_texto})

        return (
            historico_chat,
            ficheiro_audio,
            atualizar_painel_visual(),
            state_formulario["Nome Completo"],
            state_formulario["NIF / Documento"],
            state_formulario["Contacto Telefónico"],
            state_formulario["Morada / Localização"],
            state_formulario["Descrição do Pedido"]
        )
    except Exception as e:
        erro_msg = f"Lamento, ocorreu um erro ao processar o seu áudio: {str(e)}"
        historico_chat.append({"role": "user", "content": "🎤 [Áudio de Voz]"})
        historico_chat.append({"role": "assistant", "content": erro_msg})
        return historico_chat, None, atualizar_painel_visual(), "", "", "", "", ""

def atualizar_painel_visual():
    """Renderiza o painel espelho HTML dinâmico com segurança completa de tags."""
    anexos_html = ""
    if state_formulario.get("Ficheiros Anexados"):
        anexos_html = "".join(f"<li style='color: green;'>🟢 {name} (Pronto)</li>" for name in state_formulario["Ficheiros Anexados"])
    else:
        anexos_html = "<li style='color: gray;'>⚠️ Nenhum ficheiro anexado</li>"

    painel_html = f"""
    <div style="border: 2px solid #005CA9; border-radius: 8px; padding: 15px; background-color: #f8f9fa; color: #333333; font-family: sans-serif;">
        <h3 style="color: #005CA9; margin-top: 0; font-family: sans-serif;">🖥️ Espelho do Formulário Oficial (Em Tempo Real)</h3>
        <p style="font-size: 11px; color: #5f6368; margin-top: -5px;">Os dados abaixo são preenchidos automaticamente pela conversa ativa com o M.E.L.</p>
        <hr style="border-top: 1px solid #dadce0;">
        <table style="width:100%; text-align:left; border-collapse: collapse; color: #333333; font-size: 13px;">
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; width: 35%; color: #005CA9;">Requerente:</th><td style="padding: 8px 0;">{state_formulario.get('Nome Completo', 'Não preenchido')}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; color: #005CA9;">NIF:</th><td style="padding: 8px 0;">{state_formulario.get('NIF / Documento', 'Não preenchido')}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; color: #005CA9;">Contacto:</th><td style="padding: 8px 0;">{state_formulario.get('Contacto Telefónico', 'Não preenchido')}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; color: #005CA9;">Localização:</th><td style="padding: 8px 0;">{state_formulario.get('Morada / Localização', 'Não preenchido')}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; color: #005CA9;">Pedido:</th><td style="padding: 8px 0;">{state_formulario.get('Descrição do Pedido', 'Não preenchido')}</td></tr>
        </table>
        <h4 style="margin-bottom: 5px; color: #005CA9; font-family: sans-serif;">📂 Ficheiros:</h4>
        <ul style="margin-top: 5px; padding-left: 20px; font-size: 13px;">
            {anexos_html}
        </ul>
    </div>
    """
    return painel_html

In [43]:
# =========================================================================
# CÉLULA DE INTERFACE UNIFICADA DO GRADIO CORRIGIDA (EVENTOS COMPATÍVEIS)
# =========================================================================
import gradio as gr
import time

GUIMARAES_CSS = """
body {
    background-color: #f4f6f9 !important;
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif !important;
}
.gradio-container {
    max-width: 1350px !important;
    margin: auto !important;
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.05) !important;
    border-radius: 12px !important;
    background-color: #ffffff !important;
    padding: 20px !important;
}
.primary-btn, button.primary {
    background: linear-gradient(135deg, #005CA9 0%, #003F7A 100%) !important;
    color: white !important;
    border: none !important;
    border-radius: 6px !important;
    font-weight: bold !important;
}
.primary-btn:hover, button.primary:hover {
    background: linear-gradient(135deg, #003F7A 0%, #00254D 100%) !important;
    box-shadow: 0 4px 12px rgba(0, 92, 169, 0.3) !important;
}
"""

with gr.Blocks(title="Balcão Único Inteligente - Guimarães") as demo_global_v2:

    # Header Institucional
    gr.HTML("""
    <div style="display: flex; align-items: center; gap: 15px; border-bottom: 3px solid #005CA9; padding-bottom: 15px; margin-bottom: 20px;">
        <div style="background-color: #005CA9; color: white; padding: 10px 18px; font-weight: 900; font-size: 24px; border-radius: 8px; font-family: sans-serif;">
            Guimarães
        </div>
        <div>
            <h1 style="margin: 0; color: #333333; font-size: 22px; font-weight: 700; font-family: sans-serif;">Balcão Único Inteligente & Acessível</h1>
            <p style="margin: 0; color: #666666; font-size: 13px; font-family: sans-serif;">Falar com o Município sem burocracia · Preenchimento Automático por Inteligência Artificial</p>
        </div>
    </div>
    """)

    with gr.Row():
        # LADO ESQUERDO: Conversa Multimodal (Texto / Voz)
        with gr.Column(scale=6):
            gr.Markdown("### 💬 1. Fale ou Escreva ao Assistente M.E.L.")

            # Chatbot de texto integrado
            chat_texto = gr.Chatbot(label="Canal de Conversação")
            msg_input = gr.Textbox(placeholder="Escreva aqui a sua questão ou envie dados do formulário...", label="A sua Mensagem")

            # Linha de envio por áudio
            gr.Markdown("#### 🎤 Preferência por Voz?")
            input_audio = gr.Audio(label="Grave utilizando o Microfone", type="filepath", sources=["microphone"])
            botao_enviar_voz = gr.Button("🎤 Enviar Mensagem por Voz", variant="primary")
            output_audio_resposta = gr.Audio(label="Resposta de Voz", autoplay=True)

        # LADO DIREITO: Painel de Validação e Envio do Formulário dinâmico
        with gr.Column(scale=6):
            gr.Markdown("### 📋 2. Formulário Inteligente Auto-Preenchido")

            # Painel espelho dinâmico em HTML
            espelho_html = gr.HTML(value=atualizar_painel_visual())

            gr.Markdown("#### 🔍 Edição Manual do Formulário")
            in_nome = gr.Textbox(label="Nome Completo do Requerente", placeholder="Detectado automaticamente...")
            in_nif = gr.Textbox(label="NIF / Identificação", placeholder="Detectado automaticamente...")
            in_tel = gr.Textbox(label="Contacto Telefónico", placeholder="Detectado automaticamente...")
            in_morada = gr.Textbox(label="Morada / Localização", placeholder="Detectado automaticamente...")
            in_desc = gr.TextArea(label="Descrição do Pedido", placeholder="Detectado automaticamente...")
            in_files = gr.File(label="Anexar Documentação Extra (PDF, Fotos)", file_count="multiple")

            botao_submeter = gr.Button("🚀 Validar e Submeter Candidatura Oficial", variant="primary")
            out_submissao = gr.Markdown(visible=False)

    # LIGAÇÕES DE EVENTOS COMPATÍVEIS

    def handle_text_chat(message, history):
        if history is None:
            history = []

        # Executa backend de RAG e extração de dados com segurança
        resposta, painel = chat_fn_unificado(message, history)

        # Adiciona ao histórico de conversação
        history.append((message, resposta))

        # Devolve atualizações seguras de dados para todos os campos correspondentes do estado global
        return (
            "",
            history,
            painel,
            state_formulario["Nome Completo"],
            state_formulario["NIF / Documento"],
            state_formulario["Contacto Telefónico"],
            state_formulario["Morada / Localização"],
            state_formulario["Descrição do Pedido"]
        )

    msg_input.submit(
        handle_text_chat,
        inputs=[msg_input, chat_texto],
        outputs=[msg_input, chat_texto, espelho_html, in_nome, in_nif, in_tel, in_morada, in_desc]
    )

    botao_enviar_voz.click(
        processar_conversa_com_audio,
        inputs=[input_audio, chat_texto],
        outputs=[chat_texto, output_audio_resposta, espelho_html, in_nome, in_nif, in_tel, in_morada, in_desc]
    )

    # Botão de submissão final do formulário preenchido por e-mail
    botao_submeter.click(
        submeter_com_geracao_pdf_e_email,
        inputs=[],
        outputs=out_submissao
    )

try:
    demo_global_v2.close()
except Exception:
    pass

demo_global_v2.launch(share=True, prevent_thread_lock=True, css=GUIMARAES_CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2293360ddbcaf1f056.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [44]:
# Forçar a reinicialização da interface do Gradio para carregar as funções atualizadas
try:
    demo_global_v2.close()
except Exception:
    pass

demo_global_v2.launch(share=True, prevent_thread_lock=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2293360ddbcaf1f056.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [45]:
# ==================================================
# Interface de Utilizador Unificada (Único launch() no Final)
# ==================================================

with gr.Blocks(title="Balcão Aberto Unificado - Guimarães") as demo_global:
    gr.Markdown("# 🏛️ Balcão Aberto - Município de Guimarães\n"
                "Assistente inteligente integrado com o **M.E.L. (DeepSeek)**. Obtenha planos baseados na documentação real e preencha formulários oficiais de forma acessível.")

    with gr.Tab("💬 Perguntar ao Assistente"):
        gr.ChatInterface(
            fn=chat_fn,
            examples=[
                "Como peço o cartão municipal sénior?",
                "Quanto custa a segunda via do cartão sénior?",
                "Preciso de autorização para lançar fogo?",
            ],
        )

    with gr.Tab("🎙️ Interação por Voz (Acessível)"):
        gr.Markdown("### 🎙️ Fale com o Município\nGrava a tua mensagem de voz utilizando o microfone abaixo e clica em 'Enviar Voz'. O assistente responderá por escrito e em áudio falado.")
        with gr.Row():
            with gr.Column(scale=1):
                input_audio = gr.Audio(label="Grave a sua pergunta ou dúvida", type="filepath", sources=["microphone"])
                botao_enviar_voz = gr.Button("🎤 Enviar Mensagem por Voz", variant="primary")
            with gr.Column(scale=1):
                output_chat_voz = gr.Chatbot(label="Conversa ativa por Voz")
                output_audio_resposta = gr.Audio(label="Ouvir Resposta do Assistente", autoplay=True)
        botao_enviar_voz.click(
            processar_conversa_com_audio,
            inputs=[input_audio, output_chat_voz],
            outputs=[output_chat_voz, output_audio_resposta]
        )

    with gr.Tab("📋 Planear Cenário Complexo"):
        gr.Markdown("### 📋 Planeamento de Atividades e Obras\n"
                    "Descreva a atividade que deseja iniciar no concelho de Guimarães e o M.E.L. construirá uma checklist de documentos oficiais e prazos reais.")
        situacao_in = gr.Textbox(label="A minha situação", lines=2, placeholder="Ex.: vou abrir um café no centro de Guimarães")
        botao_plano = gr.Button("Criar o meu plano", variant="primary")
        plano_out = gr.Markdown()
        pdf_out = gr.File(label="Checklist em PDF")
        botao_plano.click(planear_mel_deepseek, inputs=situacao_in, outputs=[plano_out, pdf_out])

    with gr.Tab("📄 Preencher Formulário (Espelho)"):
        gr.Markdown("### 🖥️ Assistente de Pré-Preenchimento e Submissão Digital\n"
                    "Insira os dados à esquerda e veja a validação atualizar-se no painel oficial à direita em tempo real.")
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("#### 📝 1. Preenchimento Interativo")
                in_nome = gr.Textbox(label="Nome Completo do Requerente", placeholder="Ex: João Silva")
                in_nif = gr.Textbox(label="NIF / Número de Identificação", placeholder="Ex: 123456789")
                in_tel = gr.Textbox(label="Contacto de Telefone", placeholder="Ex: 912345678")
                in_morada = gr.Textbox(label="Morada / Localização do Pedido", placeholder="Ex: Largo Cónego José Maria Gomes")
                in_desc = gr.TextArea(label="Informação Adicional / Descrição da Ocorrência", placeholder="Detalhes...")
                in_files = gr.File(label="Arraste fotos ou PDFs necessários", file_count="multiple")
            with gr.Column(scale=1):
                espelho_html = gr.HTML(value=atualizar_espelho("", "", "", "", "", None))
                campos_input = [in_nome, in_nif, in_tel, in_morada, in_desc, in_files]
                for campo in campos_input:
                    campo.change(atualizar_espelho, inputs=campos_input, outputs=espelho_html)
                botao_submeter = gr.Button("🚀 Enviar Candidatura Pronta aos Serviços Municipais", variant="primary")
                out_submissao = gr.Markdown(visible=False)
                botao_submeter.click(submeter_com_geracao_pdf_e_email, inputs=[], outputs=out_submissao)



# Inicializar a nova versão redesenhada do Balcão Aberto
demo_global_v2.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2293360ddbcaf1f056.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
